# scale_report

> Phase 14 report: `trustSummaries` (`TRUST_PLAN`: loss recovered in the 2 x 2 of trust information, the publicity and cost effects and their difference, mechanism columns), `scaleSummaries` (`SCALE_PLAN`: per N the request board's gap closed, the track record's loss recovered and the baseline), `writeTrust`, `writeScale` and `figScale`. All findings are exploratory.

In [ ]:
#| default_exp scale_report

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import re

import matplotlib

matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

from explRL.bench_grid import rowKey
from explRL.fix_report import _bySeed, fixLogMetrics
from explRL.fix_stats import bootStat, gapClosed, holmFamily, lossRecovered
from explRL.llm_figures import _save
from explRL.llm_report import NAN, _f, _sel, _stat

RECORD = re.compile(r', record (?:(\d+)/(\d+) true|none)\)')
TRUST = dict(record=('public', 'free'), recordPrivate=('private', 'free'), verify=('private', 'costly'),
             verifyPublic=('public', 'costly'))                                       # arm -> (who sees it, its cost)
MECH = ('welfare', 'welfareHonest', 'pooledShare', 'invalid')


def trustLogMetrics(records):
    """fixLogMetrics per row key plus, from the observations, the track-record entries agents saw: recordEntries,
    recordChecked (mean checked shares behind an entry; 'record none' counts 0) and recordNone (share of 'none')."""
    out = fixLogMetrics(records)
    for x in records:
        if x.get('kind', 'turn') == 'turn' and x['key'] in out:
            d = out[x['key']]
            for m in RECORD.finditer(x.get('obs', '')):
                d['recordEntries'] = d.get('recordEntries', 0) + 1
                d['_checked'] = d.get('_checked', 0) + int(m.group(2) or 0)
                d['_none'] = d.get('_none', 0) + (m.group(2) is None)
    for d in out.values():
        n = d.setdefault('recordEntries', 0)
        d['recordChecked'], d['recordNone'] = (d.pop('_checked', 0) / n, d.pop('_none', 0) / n) if n else (NAN, NAN)
    return out


def _loss(rows, arm, N, clean=None):
    "{seed: welfareHonest without liars - with them} for an arm at N (clean: the level of the clean cells, else the arm)."
    c, l = (_bySeed(_sel(rows, level=lv, N=N), 'welfareHonest') for lv in (clean or arm, arm + '|liar'))
    return {s: c[s] - l[s] for s in sorted(set(c) & set(l))}


def _mech(rs, logm, cols=('checks', 'checkedFalse', 'recordChecked', 'recordNone')):
    m = {k: _stat(_f(r.get(k, NAN)) for r in rs)['mean'] for k in MECH}
    m.update({k: _stat(logm.get(rowKey(r), {}).get(k, NAN) for r in rs)['mean'] for k in cols})
    return dict(m, runs=len(rs))


def trustSummaries(rows13, rows14, logMetrics=None, study13='fixA', study14='trustA'):
    """TRUST_PLAN: loss recovered per arm of the 2 x 2 (record, verify, baseline: Phase 13's rows; recordPrivate,
    verifyPublic: Phase 14's) and its no-liar cost; the publicity and cost effects and their difference (bootstrap over
    the seeds every arm has, resampled together); Holm over the two new arms; mechanism columns per cell."""
    r13, r14, logm = _sel(list(rows13), study=study13, N=4), _sel(list(rows14), study=study14, N=4), logMetrics or {}
    src = dict(baseline=r13, record=r13, verify=r13, recordPrivate=r14, verifyPublic=r14)
    L = {a: _loss(rs, a, 4) for a, rs in src.items()}
    S = dict(arms={}, cost={}, effects={}, mechanism={})
    for a in TRUST:
        if L[a] and L['baseline']:
            S['arms'][a] = dict(lossRecovered(L[a], L['baseline']), arm=a, kind='affordance', public=TRUST[a][0] == 'public',
                                free=TRUST[a][1] == 'free', new=a in ('recordPrivate', 'verifyPublic'))
        w, b = _bySeed(_sel(src[a], level=a), 'welfare'), _bySeed(_sel(r13, level='baseline'), 'welfare')
        if w:
            S['cost'][a] = _stat(w[s] - b[s] for s in sorted(set(w) & set(b)))
    holmFamily(S['arms'], [a for a in ('recordPrivate', 'verifyPublic') if a in S['arms']])
    seeds = sorted(set.intersection(*(set(v) for v in L.values()))) if all(L.values()) else []
    if seeds:
        A = {a: np.array([L[a][s] for s in seeds], float) for a in L}
        lr = lambda a, i: 1 - A[a][i].mean() / A['baseline'][i].mean() if A['baseline'][i].mean() != 0 else NAN
        pub = lambda i: (lr('record', i) + lr('verifyPublic', i)) / 2 - (lr('recordPrivate', i) + lr('verify', i)) / 2
        cost = lambda i: (lr('record', i) + lr('recordPrivate', i)) / 2 - (lr('verify', i) + lr('verifyPublic', i)) / 2
        for k, f in (('publicity', pub), ('cost', cost), ('difference', lambda i: pub(i) - cost(i))):
            S['effects'][k] = dict(effect=float(f(np.arange(len(seeds)))), ci=bootStat(f, len(seeds)), n=len(seeds))
    for lv in sorted({r['level'] for r in r14} | {r['level'] for r in r13 if r['level'].split('|')[0] in src}):
        S['mechanism'][lv] = _mech(_sel(src[lv.split('|')[0]], level=lv), logm)
    return S


def scaleSummaries(rows, study='scaleB'):
    """SCALE_PLAN: per N, request's pooled-share gap closed (clean, ceiling 1) and record's loss recovered (honest loss
    of N // 4 liars; record's clean reference is its own clean cells if run, else the baseline's at the same N and seed),
    Holm over N within each arm; the baseline's pooled share and honest loss per N; per arm the trend across N."""
    rs = _sel(list(rows), study=study)
    S = dict(request={}, record={}, baseline={}, trend=dict(request={}, record={}))
    for N in sorted({int(_f(r['N'])) for r in rs}):
        at = _sel(rs, N=N)
        base, req = _bySeed(_sel(at, level='baseline'), 'pooledShare'), _bySeed(_sel(at, level='request'), 'pooledShare')
        if base and req:
            S['request'][f'request|{N}'] = dict(gapClosed(req, base, 1.0), arm='request', N=N)
        bl = _loss(at, 'baseline', N)
        own = bool(_sel(at, level='record'))
        rl = _loss(at, 'record', N, None if own else 'baseline')
        if bl and rl:
            S['record'][f'record|{N}'] = dict(lossRecovered(rl, bl), arm='record', N=N, cleanRef='record' if own else 'baseline')
        S['baseline'][N] = dict(pooledShare=_stat(base.values()), honestLoss=_stat(bl.values()),
                                welfareHonest=_stat(_bySeed(_sel(at, level='baseline'), 'welfareHonest').values()))
    for arm, stat in (('request', 'gap'), ('record', 'gap')):
        holmFamily(S[arm], list(S[arm]))
        S['trend'][arm] = {v['N']: v[stat] for v in S[arm].values()}
    return S


def _g(x, d=2):
    return 'n/a' if x is None or x != x else f'{x:.{d}f}'


def _head(title, meta, plan):
    return [f"# {title} ({meta['mode']})", '', f'Exploratory, pre-specified in `{plan}`.', '',
            f"- Plan hash `{meta['planHash']}`, git commit `{meta['commit']}`; model {meta['model']}; prices checked "
            f"{meta['pricesChecked']}; cost ${meta['cost']:.2f}; total API spend ${meta['spent']:.2f} of the ${meta['cap']:.0f} "
            f"cap; calls {meta['calls']}; wall time {meta['wall']:.0f} s"]


def _row(k, v):
    return (f"| {k} | {_g(v['gap'])} [{_g(v['ci'][0])}, {_g(v['ci'][1])}] | {_g(v['base'])} | {_g(v['diff']['mean'], 3)} | "
            f"{v['n']} | {_g(v['p'], 3)} | {_g(v['pHolm'], 3) if v['primary'] else 'Phase 13'} |")


def _mtab(mech):
    cols = sorted({c for v in mech.values() for c in v if c != 'runs'})
    return (['', 'Mechanism (means per cell)', '', '| cell | runs | ' + ' | '.join(cols) + ' |', '|---|---|' + '---|' * len(cols)] +
            [f"| {k} | {v['runs']} | " + ' | '.join(_g(v.get(c, NAN)) for c in cols) + ' |' for k, v in mech.items()]) if mech else []


def writeTrust(path, S, meta):
    "report.md for Part A: provenance, the 2 x 2's loss recovered, the publicity and cost effects, no-liar costs, mechanism."
    L = _head('Phase 14, Part A: publicity or verification?', meta, 'TRUST_PLAN')
    L += ['', '## Loss recovered per arm (one liar, N 4; paired on board seed)', '',
          '| arm | loss recovered [95% CI] | baseline honest loss | mean paired difference | n | p | Holm p |',
          '|---|---|---|---|---|---|---|'] + [_row(f"{k} ({'/'.join(TRUST[k])})", v) for k, v in S['arms'].items()]
    L += ['', '## Effects (in loss recovered; bootstrap over seeds, the same seeds for every arm)', '',
          '| effect | estimate [95% CI] | n |', '|---|---|---|']
    L += [f"| {k} | {_g(v['effect'])} [{_g(v['ci'][0])}, {_g(v['ci'][1])}] | {v['n']} |" for k, v in S['effects'].items()]
    L += ['', "## No-liar cost (arm's welfare minus baseline's, paired on seed)", '', '| arm | difference [95% CI] | n |', '|---|---|---|']
    L += [f"| {k} | {_g(v['mean'], 3)} [{_g(v['ci'][0], 3)}, {_g(v['ci'][1], 3)}] | {v['n']} |" for k, v in S['cost'].items()]
    path.write_text('\n'.join(L + _mtab(S['mechanism'])) + '\n')
    return path


def writeScale(path, S, meta):
    "report.md for Part B: provenance, per N the request's gap closed and the record's loss recovered, the baseline per N."
    L = _head('Phase 14, Part B: do the fixes scale?', meta, 'SCALE_PLAN')
    for arm, title in (('request', "request: pooled-share gap closed (clean, ceiling 1)"),
                       ('record', 'record: loss recovered (a quarter of the agents liars)')):
        L += ['', f'## {title}', '', '| arm, N | estimate [95% CI] | baseline | mean paired difference | n | p | Holm p (over N) |',
              '|---|---|---|---|---|---|---|'] + [_row(k, v) for k, v in S[arm].items()]
    L += ['', '## Baseline per N', '', '| N | pooled share (clean) | honest loss | honest welfare (clean) |', '|---|---|---|---|']
    L += [f"| {N} | " + ' | '.join(f"{_g(v[k]['mean'], 3)} [{_g(v[k]['ci'][0], 3)}, {_g(v[k]['ci'][1], 3)}]"
                                    for k in ('pooledShare', 'honestLoss', 'welfareHonest')) + ' |' for N, v in S['baseline'].items()]
    path.write_text('\n'.join(L) + '\n')
    return path


def _bar(ax, items, label):
    for y, (k, v) in enumerate(items):
        x = v.get('gap', v.get('effect'))
        lo, hi = (np.nan_to_num(c, nan=x) for c in v['ci'])
        ax.errorbar([x], [y], xerr=[[max(x - lo, 0)], [max(hi - x, 0)]], fmt='o', capsize=3)
    ax.set_yticks(range(len(items)), [k for k, _ in items], fontsize=8)
    ax.axvline(0, color='k', lw=0.5)
    ax.invert_yaxis()
    ax.set(xlabel=label)


def figScale(out, trust=None, scale=None):
    "trust.png (loss recovered per arm and the effects) and scale.png (request's gap and record's recovery vs N)."
    paths = []
    if trust and trust['arms']:
        fig, axs = plt.subplots(1, 2, figsize=(9, 3.2))
        _bar(axs[0], list(trust['arms'].items()), 'loss recovered')
        _bar(axs[1], list(trust['effects'].items()), 'effect (loss recovered)')
        paths.append(_save(fig, out / 'trust.png'))
    if scale and (scale['request'] or scale['record']):
        fig, ax = plt.subplots(figsize=(5, 3.5))
        for arm, lab in (('request', 'request: gap closed'), ('record', 'record: loss recovered')):
            v = list(scale[arm].values())
            if v:
                ax.errorbar([x['N'] for x in v], [x['gap'] for x in v], capsize=3, marker='o', label=lab,
                            yerr=np.abs(np.nan_to_num(np.array([x['ci'] for x in v]).T - [x['gap'] for x in v])))
        ax.axhline(0, color='k', lw=0.5)
        ax.set(xticks=sorted({x['N'] for a in ('request', 'record') for x in scale[a].values()}), xlabel='N')
        ax.legend(fontsize=7)
        paths.append(_save(fig, out / 'scale.png'))
    return paths

## Tests

In [ ]:
import json
import tempfile
from pathlib import Path
from explRL.scale_report import *
from explRL.bench_grid import rowKey

def row(study, level, N, seed, **m):
    return dict(study=study, factor=study, level=level, channel='blackboard', budget='turns', N=N, seed=seed, episode=0,
                **{**dict(welfare=1.0, welfareHonest=1.0, pooledShare=0.5, invalid=0), **m})
# Part A on synthetic rows with known effects: each arm's liar loss is baseline's x (1 - LR), seed by seed
LR = dict(record=0.6, recordPrivate=0.2, verify=0.0, verifyPublic=0.4)
seeds = range(900, 910)
loss = {s: 2.0 + 0.3 * (s % 4) for s in seeds}
r13, r14 = [], []
for s in seeds:
    for a, rs in (('baseline', r13), ('record', r13), ('verify', r13), ('recordPrivate', r14), ('verifyPublic', r14)):
        st, clean = ('fixA' if rs is r13 else 'trustA'), 10.0 + s % 3
        rs.append(row(st, a, 4, s, welfareHonest=clean, welfare=clean + 0.1 * (a == 'record')))
        rs.append(row(st, a + '|liar', 4, s, welfareHonest=clean - loss[s] * (1 - LR.get(a, 0.0))))
r13.append(row('fixA', 'request', 8, 900))                                          # other cells are ignored
S = trustSummaries(r13, r14)
assert set(S['arms']) == set(LR) and all(abs(S['arms'][a]['gap'] - LR[a]) < 1e-9 for a in LR)
assert S['arms']['recordPrivate']['primary'] and S['arms']['verifyPublic']['primary'] and not S['arms']['record']['primary']
assert S['arms']['record']['pHolm'] != S['arms']['record']['pHolm']                 # Phase 13 arms: no new test
assert S['arms']['verifyPublic']['public'] and not S['arms']['verifyPublic']['free'] and S['arms']['recordPrivate']['free']
E = S['effects']
for k, v in (('publicity', (0.6 + 0.4) / 2 - (0.2 + 0.0) / 2), ('cost', (0.6 + 0.2) / 2 - (0.0 + 0.4) / 2)):
    assert abs(E[k]['effect'] - v) < 1e-9 and all(abs(c - v) < 1e-9 for c in E[k]['ci']) and E[k]['n'] == 10, k
assert abs(E['difference']['effect'] - 0.2) < 1e-9
assert abs(S['cost']['record']['mean'] - 0.1) < 1e-9 and S['cost']['verifyPublic']['mean'] == 0 and set(S['cost']) == set(LR)
# a noisy arm: the bootstrap interval opens around the estimate, the same seeds resampled for every arm
noisy = [dict(r, welfareHonest=r['welfareHonest'] - (0.5 if r['seed'] % 2 else -0.5)) if r['level'] == 'verifyPublic|liar' else r for r in r14]
e = trustSummaries(r13, noisy)['effects']['publicity']
assert e['ci'][0] < e['effect'] < e['ci'][1] and abs(e['effect'] - 0.4) < 1e-9          # mean loss unchanged (5 up, 5 down)
# seeds missing from one arm drop out of the effects (all arms share the seeds)
assert trustSummaries(r13, [r for r in r14 if r['seed'] != 909])['effects']['publicity']['n'] == 9
# mechanism: checks, checks found false and the checked shares behind the track records shown
k = rowKey(row('trustA', 'recordPrivate|liar', 4, 900))
k2 = rowKey(row('trustA', 'verifyPublic|liar', 4, 900))
recs = [dict(key=k, kind='turn', valid=True, report=None, response='{"action": "wait"}',
             obs='T1: x mod 3 = 1 (agent 0, 2 turns ago, record 1/3 true); T2: x mod 5 = 2 (agent 2, 1 turns ago, record none)'),
        dict(key=k, kind='turn', valid=True, report=None, response='{"action": "wait"}',
             obs='T1: x mod 3 = 1 (agent 0, 3 turns ago, record 0/1 true)'),
        dict(key=k2, kind='turn', valid=True, report=None, response='{"action": "check"}',
             obs='Your check: T1 x mod 3 = 1 (from agent 0) is false.\nChecked by agent 2: T2 x mod 5 = 2 (from agent 1) is true.')]
m = trustLogMetrics(recs)
assert m[k]['recordEntries'] == 3 and abs(m[k]['recordChecked'] - 4 / 3) < 1e-9 and abs(m[k]['recordNone'] - 1 / 3) < 1e-9
assert m[k2]['checks'] == 1 and m[k2]['checkedFalse'] == 1 and m[k2]['recordEntries'] == 0 and m[k2]['recordChecked'] != m[k2]['recordChecked']
S = trustSummaries(r13, r14, m)
assert S['mechanism']['recordPrivate|liar']['recordChecked'] == 4 / 3 and S['mechanism']['verifyPublic|liar']['checks'] == 1
assert set(S['mechanism']) == {a + x for a in ('baseline', *LR) for x in ('', '|liar')} and S['mechanism']['record']['runs'] == 10

# Part B on synthetic rows: request closes -0.1 of the gap and record recovers half the loss at every N
rows = []
for N in (8, 16, 32):
    for s in range(900, 905):
        b, cl, B = 0.4 + 0.02 * (s % 3), 10.0 + s % 2, 1.0 + N / 8 + 0.1 * (s % 4)
        rows += [row('scaleB', 'baseline', N, s, pooledShare=b, welfareHonest=cl), row('scaleB', 'request', N, s, pooledShare=b - 0.1 * (1 - b)),
                 row('scaleB', 'baseline|liar', N, s, welfareHonest=cl - B), row('scaleB', 'record|liar', N, s, welfareHonest=cl - 0.5 * B)]
T = scaleSummaries(rows + [row('trustA', 'request', 8, 900, pooledShare=1.0)])
assert set(T['request']) == {'request|8', 'request|16', 'request|32'} and set(T['record']) == {'record|8', 'record|16', 'record|32'}
assert all(abs(v['gap'] + 0.1) < 1e-9 for v in T['request'].values()) and all(abs(v['gap'] - 0.5) < 1e-9 for v in T['record'].values())
assert all(v['cleanRef'] == 'baseline' and v['primary'] and v['n'] == 5 for v in T['record'].values())
assert all(v['primary'] and v['pHolm'] == v['pHolm'] for v in T['request'].values())
assert T['trend']['record'] == {8: 0.5, 16: 0.5, 32: 0.5} or all(abs(x - 0.5) < 1e-9 for x in T['trend']['record'].values())
assert abs(T['baseline'][16]['honestLoss']['mean'] - (3 + 0.1 * np.mean([s % 4 for s in range(900, 905)]))) < 1e-9
assert abs(T['baseline'][8]['pooledShare']['mean'] - np.mean([0.4 + 0.02 * (s % 3) for s in range(900, 905)])) < 1e-9
# with record's own clean cells, they are its clean reference
own = rows + [row('scaleB', 'record', N, s, welfareHonest=11.0 + s % 2) for N in (8, 16, 32) for s in range(900, 905)]
own = [dict(r, welfareHonest=r['welfareHonest'] + 1) if r['level'] == 'record|liar' else r for r in own]
assert all(v['cleanRef'] == 'record' and abs(v['gap'] - 0.5) < 1e-9 for v in scaleSummaries(own)['record'].values())

# reports and figures
meta = dict(mode='test', planHash='abc', commit='x', model='claude-haiku-5-5', pricesChecked='2026-10-12', cost=0.0, spent=0.0,
            cap=150.0, calls=0, wall=0.0)
with tempfile.TemporaryDirectory() as d:
    d = Path(d)
    t = writeTrust(d / 't.md', S, meta).read_text()
    assert '| recordPrivate (private/free) | 0.20 [0.20, 0.20]' in t and '| publicity | 0.40 [0.40, 0.40] | 10 |' in t
    assert 'Phase 13 |' in t and 'Mechanism' in t and 'TRUST_PLAN' in t
    sc = writeScale(d / 's.md', T, meta).read_text()
    assert '| request|8 | -0.10 [-0.10, -0.10]' in sc and '| record|32 | 0.50' in sc and '## Baseline per N' in sc and 'SCALE_PLAN' in sc
    ps = figScale(d, S, T)
    assert [p.name for p in ps] == ['trust.png', 'scale.png'] and all(p.stat().st_size > 0 for p in ps)
    assert figScale(d, None, scaleSummaries([])) == []
json.dumps(S, default=str), json.dumps(T, default=str)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()